# Chapter 4: Collective Patterns
This notebook covers the Triton implementations found in Chapter 4, specifically reductions, atomic operations, Layer/RMS Norm, Softmax, sorting, and nucleus sampling.
The code is fully runnable sequentially.


In [ ]:
import torch
import triton
import triton.language as tl

torch.manual_seed(0)
print("Dependencies loaded. Triton and PyTorch ready.")


In [ ]:
@triton.jit
def row_sum_max_kernel(x_ptr, out_sum_ptr, out_max_ptr,
                       stride_x, n_rows, 
                       N_COLS: tl.constexpr,
                       BLOCK_SIZE: tl.constexpr):
    row = tl.program_id(0)
    x_row_ptr = x_ptr + row * stride_x

    acc_sum = tl.zeros([BLOCK_SIZE], dtype=tl.float32) 
    acc_max = tl.full([BLOCK_SIZE], float("-inf"), dtype=tl.float32) 
    neg_inf = tl.full([BLOCK_SIZE], float("-inf"), dtype=tl.float32)

    for off in tl.static_range(0, N_COLS, BLOCK_SIZE): 
        cols = off + tl.arange(0, BLOCK_SIZE)
        mask = cols < N_COLS

        x = tl.load(x_row_ptr + cols, mask=mask, other=0.0).to(tl.float32)

        acc_sum += x
        acc_max = tl.maximum(acc_max, tl.where(mask, x, neg_inf)) 

    row_sum = tl.sum(acc_sum, axis=0) 
    row_max = tl.max(acc_max, axis=0)

    tl.store(out_sum_ptr + row, row_sum)
    tl.store(out_max_ptr + row, row_max)


def row_sum_max(x: torch.Tensor, block_size: int = 1024):
    assert x.is_cuda and x.ndim == 2
    n_rows, n_cols = x.shape

    out_sum = torch.empty((n_rows,), device=x.device, dtype=torch.float32)
    out_max = torch.empty((n_rows,), device=x.device, dtype=torch.float32)

    grid = (n_rows,)
    row_sum_max_kernel[grid](
        x, out_sum, out_max,
        x.stride(0), n_rows,
        N_COLS=n_cols,
        BLOCK_SIZE=block_size,
        num_warps=4,
    )
    return out_sum, out_max

# End-to-end execution
x_test = torch.randn((10, 1024), device='cuda')
out_sum, out_max = row_sum_max(x_test, block_size=1024)
print("Row sum shape:", out_sum.shape)
print("Row max shape:", out_max.shape)
print("Row Sum and Max correctly executed.")


In [ ]:
@triton.jit
def partial_sum_kernel(x_ptr, partial_ptr, n_elements,
                       BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements

    x = tl.load(x_ptr + offsets, mask=mask, other=0.0)
    part = tl.sum(x, axis=0, dtype=tl.float32)
    tl.store(partial_ptr + pid, part)


@triton.jit
def final_sum_kernel(partial_ptr, out_ptr, n_partials,
                     BLOCK_SIZE: tl.constexpr):
    offsets = tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_partials

    x = tl.load(partial_ptr + offsets, mask=mask, other=0.0)
    total = tl.sum(x, axis=0, dtype=tl.float32)
    tl.store(out_ptr, total)


def sum_two_pass(x: torch.Tensor, block_size: int = 1024):
    assert x.is_cuda and x.ndim == 1
    n = x.numel()
    grid0 = (triton.cdiv(n, block_size),)
    partial = torch.empty((grid0[0],), device=x.device, dtype=torch.float32)
    partial_sum_kernel[grid0](x, partial, n, BLOCK_SIZE=block_size)
    assert partial.numel() <= block_size
    out = torch.empty((1,), device=x.device, dtype=torch.float32)
    final_sum_kernel[(1,)](partial, out, partial.numel(), BLOCK_SIZE=block_size)

    return out[0]

@triton.jit
def atomic_sum_kernel(x_ptr, out_ptr, n_elements,
                      BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements

    x = tl.load(x_ptr + offsets, mask=mask, other=0.0)
    part = tl.sum(x, axis=0, dtype=tl.float32)
    tl.atomic_add(out_ptr, part)


def sum_atomic(x: torch.Tensor, block_size: int = 1024):
    assert x.is_cuda and x.ndim == 1
    n = x.numel()

    out = torch.zeros((1,), device=x.device, dtype=torch.float32)
    grid = (triton.cdiv(n, block_size),)
    atomic_sum_kernel[grid](x, out, n, BLOCK_SIZE=block_size)
    return out[0]

# Execution
x_1d = torch.randn((4096,), device='cuda')
total_two_pass = sum_two_pass(x_1d)
total_atomic = sum_atomic(x_1d)
print("Two Pass Sum:", total_two_pass.item())
print("Atomic Sum:", total_atomic.item())
print("Torch Sum (for ref):", x_1d.sum().item())


In [ ]:
@triton.jit
def rms_norm_1d_kernel(x_ptr, gamma_ptr, y_ptr,
                       N: tl.constexpr, eps,
                       BLOCK_SIZE: tl.constexpr):
    sumsq = tl.zeros([BLOCK_SIZE], dtype=tl.float32) 

    for off in range(0, N, BLOCK_SIZE):  
        idx = off + tl.arange(0, BLOCK_SIZE)
        mask = idx < N
        x_val = tl.load(x_ptr + idx, mask=mask, other=0.0).to(tl.float32)
        sumsq += x_val * x_val

    total = tl.sum(sumsq, axis=0) 
    inv_rms = 1.0 / tl.sqrt(total / N + eps) 

    # Pass 2: normalize and apply scale.
    for off in range(0, N, BLOCK_SIZE):
        idx = off + tl.arange(0, BLOCK_SIZE)
        mask = idx < N
        x_val = tl.load(x_ptr + idx, mask=mask, other=0.0).to(tl.float32)
        g_val = tl.load(gamma_ptr + idx, mask=mask, other=1.0).to(tl.float32)
        y_val = x_val * inv_rms * g_val 
        tl.store(y_ptr + idx, y_val, mask=mask)


def rms_norm_1d(x: torch.Tensor, gamma: torch.Tensor,
                eps: float = 1e-6, block_size: int = 1024):
    assert x.is_cuda and x.ndim == 1
    assert gamma.is_cuda and gamma.shape == x.shape
    n = x.numel()

    y = torch.empty_like(x, dtype=torch.float32)
    rms_norm_1d_kernel[(1,)](
        x, gamma, y,
        N=n, eps=eps, BLOCK_SIZE=block_size,
        num_warps=4,
    )
    return y

@triton.jit
def rmsnorm_kernel(x_ptr, gamma_ptr, y_ptr,
                   stride_x, stride_y,
                   eps: tl.constexpr,
                   N: tl.constexpr,
                   BLOCK_SIZE: tl.constexpr):
    row = tl.program_id(0)

    x_row_ptr = x_ptr + row * stride_x
    y_row_ptr = y_ptr + row * stride_y

    sumsq = tl.zeros([BLOCK_SIZE], dtype=tl.float32)

    # Pass 1: accumulate sum of squares for this row.
    for off in tl.static_range(0, N, BLOCK_SIZE):
        cols = off + tl.arange(0, BLOCK_SIZE)
        mask = cols < N
        x = tl.load(x_row_ptr + cols, mask=mask, other=0.0).to(tl.float32)
        sumsq += x * x

    total = tl.sum(sumsq, axis=0)
    inv_rms = tl.rsqrt(total / N + eps)

    # Pass 2: normalize and apply gamma.
    for off in tl.static_range(0, N, BLOCK_SIZE):
        cols = off + tl.arange(0, BLOCK_SIZE)
        mask = cols < N

        x = tl.load(x_row_ptr + cols, mask=mask, other=0.0).to(tl.float32)
        g = tl.load(gamma_ptr + cols, mask=mask, other=1.0).to(tl.float32)

        y = x * inv_rms * g
        tl.store(y_row_ptr + cols, y, mask=mask)


def rmsnorm(x: torch.Tensor, gamma: torch.Tensor,
            eps: float = 1e-6, block_size: int = 1024):
    assert x.is_cuda and x.ndim == 2
    assert gamma.is_cuda and gamma.ndim == 1
    assert x.shape[1] == gamma.shape[0]

    n_rows, n_cols = x.shape
    y = torch.empty_like(x)

    grid = (n_rows,)
    rmsnorm_kernel[grid](
        x, gamma, y,
        x.stride(0), y.stride(0),
        eps=eps, N=n_cols, BLOCK_SIZE=block_size,
        num_warps=4,
    )
    return y

# Execution
gamma = torch.ones((1024,), device='cuda')
y_1d = rms_norm_1d(x_1d[:1024], gamma)

x_2d = torch.randn((16, 1024), device='cuda')
y_2d = rmsnorm(x_2d, gamma)
print("RMSNorm 1D output shape:", y_1d.shape)
print("RMSNorm Batched output shape:", y_2d.shape)


In [ ]:
@triton.jit
def reduce_max_kernel(x_ptr, out_ptr, N, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < N
    x = tl.load(x_ptr + offsets, mask=mask, other=-float("inf")).to(tl.float32)
    m = tl.max(x, axis=0)
    tl.store(out_ptr + pid, m)

@triton.jit
def transform_exp_kernel(x_ptr, z_ptr, max_ptr, N, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < N
    m = tl.load(max_ptr).to(tl.float32)
    x = tl.load(x_ptr + offsets, mask=mask, other=-float("inf")).to(tl.float32)
    z = tl.exp(x - m)
    tl.store(z_ptr + offsets, z, mask=mask)

@triton.jit
def reduce_sum_kernel(z_ptr, out_ptr, N, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < N
    z = tl.load(z_ptr + offsets, mask=mask, other=0.0).to(tl.float32)
    s = tl.sum(z, axis=0)
    tl.store(out_ptr + pid, s)

@triton.jit
def div_store_kernel(z_ptr, y_ptr, sum_ptr, N, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < N
    s = tl.load(sum_ptr).to(tl.float32)
    z = tl.load(z_ptr + offsets, mask=mask, other=0.0).to(tl.float32)
    y = z / s
    tl.store(y_ptr + offsets, y, mask=mask)

def reduce_global(x, N, reduce_kernel, BLOCK_SIZE=1024): 
    cur = x
    cur_N = N
    while cur_N > 1:
        out_N = (cur_N + BLOCK_SIZE - 1) // BLOCK_SIZE
        out = torch.empty((out_N,), device=x.device, dtype=torch.float32)
        reduce_kernel[(out_N,)](cur, out, cur_N, BLOCK_SIZE=BLOCK_SIZE)
        cur = out
        cur_N = out_N
    return cur

def softmax_unfused(x: torch.Tensor, y: torch.Tensor, N: int):
    BLOCK_SIZE = 1024
    grid = ((N + BLOCK_SIZE - 1) // BLOCK_SIZE,)
    max_val = reduce_global(x, N, reduce_max_kernel) 
    z = torch.empty_like(x) 
    transform_exp_kernel[grid](x, z, max_val, N, BLOCK_SIZE=BLOCK_SIZE)
    sum_val = reduce_global(z, N, reduce_sum_kernel) 
    div_store_kernel[grid](z, y, sum_val, N, BLOCK_SIZE=BLOCK_SIZE)

@triton.jit
def softmax_fused_1d_kernel(x_ptr, y_ptr, N, BLOCK_SIZE: tl.constexpr):
    offsets = tl.arange(0, BLOCK_SIZE)
    mask = offsets < N
    x = tl.load(x_ptr + offsets, mask=mask, other=-float("inf")).to(tl.float32)
    m = tl.max(x, axis=0)
    x = x - m
    num = tl.exp(x)
    den = tl.sum(num, axis=0)
    y = num / den
    tl.store(y_ptr + offsets, y, mask=mask)

def softmax_fused_1d(x: torch.Tensor, y: torch.Tensor, N: int):
    BLOCK_SIZE = triton.next_power_of_2(N) 
    if BLOCK_SIZE > 1024:
        raise ValueError("This fused example assumes N <= 1024 so the full vector fits in one program instance.")
    
    num_warps = 8 if BLOCK_SIZE >= 512 else 4 
    softmax_fused_1d_kernel[(1,)](x, y, N, BLOCK_SIZE=BLOCK_SIZE, num_warps=num_warps)

# Execution
x_sm = torch.randn((1000,), device='cuda')
y_unfused = torch.empty_like(x_sm)
softmax_unfused(x_sm, y_unfused, N=1000)

y_fused = torch.empty_like(x_sm)
softmax_fused_1d(x_sm, y_fused, N=1000)

pytorch_sm = torch.softmax(x_sm, dim=0)
print("Unfused Softmax Max Difference:", torch.max(torch.abs(y_unfused - pytorch_sm)).item())
print("Fused Softmax Max Difference:", torch.max(torch.abs(y_fused - pytorch_sm)).item())


In [ ]:
@triton.jit
def argsort(values, n_items: tl.constexpr,
            dim: tl.constexpr = 0,
            descending: tl.constexpr = False):
    indices = tl.arange(0, n_items)
    vals_int = values.to(tl.int32, bitcast=True) 
    packed = (vals_int.to(tl.int64) << 32) | indices.to(tl.int64) 
    sorted_packed = tl.sort(packed, dim=dim,
                            descending=descending)
    sorted_indices = sorted_packed.to(tl.int32)
    sorted_vals_int = (sorted_packed >> 32).to(tl.int32)
    sorted_values = sorted_vals_int.to(tl.float32,
                                       bitcast=True)
    return sorted_values, sorted_indices

@triton.jit
def nucleus_sample_kernel(logits_ptr, random_ptr,
                          output_idx_ptr, n_vocab,
                          p_threshold,
                          BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(axis=0)
    block_start = pid * n_vocab
    offsets = block_start + tl.arange(0, BLOCK_SIZE)
    mask = offsets < (block_start + n_vocab)

    logits = tl.load(logits_ptr + offsets,
                     mask=mask, other=-float('inf'))
    max_logit = tl.max(logits, axis=0) 
    logits = logits - max_logit
    exp_logits = tl.exp(logits)
    sum_exp = tl.sum(exp_logits, axis=0)
    probs = exp_logits / sum_exp

    sorted_probs, sorted_idx = argsort(
        probs, BLOCK_SIZE, descending=True) 
    cum_probs = tl.cumsum(sorted_probs, axis=0) 

    below_threshold = cum_probs < p_threshold
    count_below = tl.sum(below_threshold, axis=0) 
    k = count_below + 1
    k = tl.minimum(k, n_vocab)

    idx_range = tl.arange(0, BLOCK_SIZE)
    cutoff_mask = idx_range == (k - 1)
    nucleus_prob_sum = tl.sum(
        tl.where(cutoff_mask, cum_probs, 0.0), axis=0) 

    rand_val = tl.load(random_ptr + pid)
    target = rand_val * nucleus_prob_sum 
    below_target = cum_probs < target
    sample_idx_in_sorted = tl.sum(below_target, axis=0)

    idx_mask = idx_range == sample_idx_in_sorted
    picked_token = tl.sum(
        tl.where(idx_mask, sorted_idx, 0), axis=0) 
    tl.store(output_idx_ptr + pid,
             picked_token.to(tl.int64))

# Wrapper and Execution
def sample_nucleus(logits: torch.Tensor, p: float = 0.8):
    assert logits.ndim == 2
    n_seq, n_vocab = logits.shape
    
    BLOCK_SIZE = triton.next_power_of_2(n_vocab)
    output_idx = torch.empty((n_seq,), device=logits.device, dtype=torch.int64)
    # Give random values between 0 and 1
    random_val = torch.rand((n_seq,), device=logits.device, dtype=torch.float32)
    
    grid = (n_seq,)
    nucleus_sample_kernel[grid](
        logits, random_val, output_idx, n_vocab, p, BLOCK_SIZE=BLOCK_SIZE, num_warps=8
    )
    return output_idx

# Test wrapper for nucleus subset
logits_sample = torch.randn((4, 1000), device='cuda')
sampled_indices = sample_nucleus(logits_sample, p=0.8)
print("Nucleus Sampling Output shape:", sampled_indices.shape)
print("Chosen token indices:", sampled_indices.tolist())
